In [1]:
# Imports and load data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────
PROCESSED_PATH = r"data\processed"

# ── Load feature matrix from EDA notebook ─────────────────────────────
feature_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'feature_matrix.csv'))

print(f" Feature matrix loaded")
print(f"  Shape:    {feature_df.shape}")
print(f"  Patients: {len(feature_df):,}")
print(f"\nColumns:\n{list(feature_df.columns)}")

 Feature matrix loaded
  Shape:    (31885, 20)
  Patients: 31,885

Columns:
['SUBJECT_ID', 'HADM_ID', 'AGE', 'GENDER', 'LOS', 'HOSPITAL_EXPIRE_FLAG', 'ADMISSION_TYPE', 'INSURANCE', 'lab_base_excess', 'lab_bicarbonate', 'lab_bun', 'lab_creatinine', 'lab_glucose', 'lab_haemoglobin', 'lab_lactate', 'lab_ph', 'lab_platelets', 'lab_potassium', 'lab_sodium', 'lab_wbc']


In [2]:
# Separate columns by type
lab_cols = [c for c in feature_df.columns if c.startswith('lab_')]

cat_cols = ['GENDER', 'ADMISSION_TYPE', 'INSURANCE']

num_cols = ['AGE', 'LOS'] + lab_cols

target_col = 'HOSPITAL_EXPIRE_FLAG'  # mortality label

id_cols = ['SUBJECT_ID', 'HADM_ID']

print(f"Lab feature columns ({len(lab_cols)}):   {lab_cols}")
print(f"Categorical columns ({len(cat_cols)}):   {cat_cols}")
print(f"Numerical columns   ({len(num_cols)}):   {num_cols}")
print(f"Target column:                           {target_col}")

Lab feature columns (12):   ['lab_base_excess', 'lab_bicarbonate', 'lab_bun', 'lab_creatinine', 'lab_glucose', 'lab_haemoglobin', 'lab_lactate', 'lab_ph', 'lab_platelets', 'lab_potassium', 'lab_sodium', 'lab_wbc']
Categorical columns (3):   ['GENDER', 'ADMISSION_TYPE', 'INSURANCE']
Numerical columns   (14):   ['AGE', 'LOS', 'lab_base_excess', 'lab_bicarbonate', 'lab_bun', 'lab_creatinine', 'lab_glucose', 'lab_haemoglobin', 'lab_lactate', 'lab_ph', 'lab_platelets', 'lab_potassium', 'lab_sodium', 'lab_wbc']
Target column:                           HOSPITAL_EXPIRE_FLAG


In [3]:
# Remove physiologically impossible values

# Clinical valid ranges — values outside these are data errors
VALID_RANGES = {
    'lab_creatinine':  (0.1,  30.0),
    'lab_sodium':      (100,  180),
    'lab_potassium':   (1.0,  12.0),
    'lab_bicarbonate': (5.0,  50.0),
    'lab_bun':         (1.0,  200.0),
    'lab_glucose':     (20,   1500),
    'lab_haemoglobin': (1.0,  25.0),
    'lab_wbc':         (0.1,  200.0),
    'lab_platelets':   (5,    2000),
    'lab_lactate':     (0.1,  30.0),
    'lab_ph':          (6.5,  8.0),
    'lab_base_excess': (-30,  30)
}

df = feature_df.copy()

print("Outlier removal:")
for col, (lo, hi) in VALID_RANGES.items():
    if col in df.columns:
        before = df[col].notna().sum()
        df[col] = df[col].where(
            (df[col] >= lo) & (df[col] <= hi), other=np.nan
        )
        after = df[col].notna().sum()
        removed = before - after
        print(f"  {col:<25} removed {removed:>4} outliers")

print(f"\n Outlier removal complete")

Outlier removal:
  lab_creatinine            removed    0 outliers
  lab_sodium                removed    0 outliers
  lab_potassium             removed    0 outliers
  lab_bicarbonate           removed    1 outliers
  lab_bun                   removed    3 outliers
  lab_glucose               removed    2 outliers
  lab_haemoglobin           removed    0 outliers
  lab_wbc                   removed    7 outliers
  lab_platelets             removed    0 outliers
  lab_lactate               removed    0 outliers
  lab_ph                    removed    0 outliers
  lab_base_excess           removed    1 outliers

 Outlier removal complete


In [4]:
# Create missingness indicator columns
# A missing lab in ICU is clinically meaningful — not random
# We preserve this information as binary columns

print("Creating missingness indicators:")
for col in lab_cols:
    indicator_col = f"{col}_missing"
    df[indicator_col] = df[col].isnull().astype(int)
    pct_missing = df[indicator_col].mean() * 100
    print(f"  {indicator_col:<35} {pct_missing:.1f}% missing")

missing_indicator_cols = [f"{c}_missing" for c in lab_cols]
print(f"\n {len(missing_indicator_cols)} missingness indicator columns created")

Creating missingness indicators:
  lab_base_excess_missing             73.6% missing
  lab_bicarbonate_missing             5.2% missing
  lab_bun_missing                     4.4% missing
  lab_creatinine_missing              4.3% missing
  lab_glucose_missing                 7.0% missing
  lab_haemoglobin_missing             5.9% missing
  lab_lactate_missing                 59.5% missing
  lab_ph_missing                      47.3% missing
  lab_platelets_missing               5.1% missing
  lab_potassium_missing               5.3% missing
  lab_sodium_missing                  5.6% missing
  lab_wbc_missing                     5.7% missing

 12 missingness indicator columns created


In [5]:
# Impute missing values with median
# Median is used instead of mean because lab distributions
# are skewed — median is more robust to extreme values

print("Imputing missing lab values with column median:")
for col in lab_cols:
    median_val = df[col].median()
    missing_count = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f"  {col:<25} median={median_val:.2f}  filled {missing_count:>4} values")

print(f"\n Imputation complete")
print(f"  Remaining nulls in lab columns: "
      f"{df[lab_cols].isnull().sum().sum()}")

Imputing missing lab values with column median:
  lab_base_excess           median=2.25  filled 23465 values
  lab_bicarbonate           median=24.00  filled 1655 values
  lab_bun                   median=17.50  filled 1393 values
  lab_creatinine            median=0.90  filled 1365 values
  lab_glucose               median=125.50  filled 2226 values
  lab_haemoglobin           median=11.10  filled 1880 values
  lab_lactate               median=1.85  filled 18976 values
  lab_ph                    median=7.38  filled 15096 values
  lab_platelets             median=205.00  filled 1614 values
  lab_potassium             median=4.10  filled 1694 values
  lab_sodium                median=139.00  filled 1780 values
  lab_wbc                   median=10.30  filled 1829 values

 Imputation complete
  Remaining nulls in lab columns: 0


In [6]:
# Encode categorical columns to numbers
# Models cannot process text — everything must be numeric

# GENDER: M=1, F=0
df['GENDER'] = df['GENDER'].map({'M': 1, 'F': 0})
print(f" GENDER encoded  — M=1, F=0")
print(f"  Value counts: {df['GENDER'].value_counts().to_dict()}")

# ADMISSION_TYPE
print(f"\nADMISSION_TYPE unique values: {df['ADMISSION_TYPE'].unique()}")
admission_dummies = pd.get_dummies(
    df['ADMISSION_TYPE'], prefix='admtype', drop_first=True
)
df = pd.concat([df, admission_dummies], axis=1)
df = df.drop(columns=['ADMISSION_TYPE'])
print(f" ADMISSION_TYPE one-hot encoded")
print(f"  New columns: {list(admission_dummies.columns)}")

# INSURANCE
print(f"\nINSURANCE unique values: {df['INSURANCE'].unique()}")
insurance_dummies = pd.get_dummies(
    df['INSURANCE'], prefix='insurance', drop_first=True
)
df = pd.concat([df, insurance_dummies], axis=1)
df = df.drop(columns=['INSURANCE'])
print(f" INSURANCE one-hot encoded")
print(f"  New columns: {list(insurance_dummies.columns)}")

# Handle any remaining NaN in new dummy columns
dummy_cols = list(admission_dummies.columns) + list(insurance_dummies.columns)
df[dummy_cols] = df[dummy_cols].fillna(0).astype(int)

 GENDER encoded  — M=1, F=0
  Value counts: {1: 18456, 0: 13429}

ADMISSION_TYPE unique values: ['EMERGENCY' 'ELECTIVE' 'URGENT']
 ADMISSION_TYPE one-hot encoded
  New columns: ['admtype_EMERGENCY', 'admtype_URGENT']

INSURANCE unique values: ['Private' 'Medicare' 'Medicaid' 'Self Pay' 'Government']
 INSURANCE one-hot encoded
  New columns: ['insurance_Medicaid', 'insurance_Medicare', 'insurance_Private', 'insurance_Self Pay']


In [7]:
# Final check and fill any remaining nulls

print("Remaining missing values before final clean:")
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
print(remaining if len(remaining) > 0 else "  None — all clean!")

# Fill any remaining nulls in numeric columns with median
for col in ['AGE', 'LOS']:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"  Filled {col} with median")

# Fill any remaining nulls in GENDER with mode
if df['GENDER'].isnull().sum() > 0:
    df['GENDER'] = df['GENDER'].fillna(df['GENDER'].mode()[0])
    print(f"  Filled GENDER with mode")

print(f"\n Final null count: {df.isnull().sum().sum()}")

Remaining missing values before final clean:
  None — all clean!

 Final null count: 0


In [8]:
# Define exactly which columns go into the model

# All columns for the generative model
model_feature_cols = (
    ['AGE', 'GENDER', 'LOS'] +
    lab_cols +
    missing_indicator_cols +
    dummy_cols
)

# The full modelling dataframe
model_df = df[model_feature_cols + [target_col]].copy()

print(f" Final modelling dataset")
print(f"  Shape:          {model_df.shape}")
print(f"  Features:       {len(model_feature_cols)}")
print(f"  Target column:  {target_col}")
print(f"  Mortality rate: {model_df[target_col].mean()*100:.1f}%")
print(f"\nAll feature columns:")
for col in model_feature_cols:
    print(f"  {col}")

 Final modelling dataset
  Shape:          (31885, 34)
  Features:       33
  Target column:  HOSPITAL_EXPIRE_FLAG
  Mortality rate: 10.6%

All feature columns:
  AGE
  GENDER
  LOS
  lab_base_excess
  lab_bicarbonate
  lab_bun
  lab_creatinine
  lab_glucose
  lab_haemoglobin
  lab_lactate
  lab_ph
  lab_platelets
  lab_potassium
  lab_sodium
  lab_wbc
  lab_base_excess_missing
  lab_bicarbonate_missing
  lab_bun_missing
  lab_creatinine_missing
  lab_glucose_missing
  lab_haemoglobin_missing
  lab_lactate_missing
  lab_ph_missing
  lab_platelets_missing
  lab_potassium_missing
  lab_sodium_missing
  lab_wbc_missing
  admtype_EMERGENCY
  admtype_URGENT
  insurance_Medicaid
  insurance_Medicare
  insurance_Private
  insurance_Self Pay


In [9]:
# Split into train and test sets
# CRITICAL — test set is locked away now and never used until
# final evaluation. This prevents data leakage.

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df[target_col]  # keeps mortality rate equal in both splits
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f" Train/test split complete")
print(f"  Train set:  {len(train_df):,} patients "
      f"({len(train_df)/len(model_df)*100:.0f}%)")
print(f"  Test set:   {len(test_df):,} patients  "
      f"({len(test_df)/len(model_df)*100:.0f}%)")
print(f"\n  Train mortality rate: "
      f"{train_df[target_col].mean()*100:.1f}%")
print(f"  Test mortality rate:  "
      f"{test_df[target_col].mean()*100:.1f}%")
print(f"\n   Test set is now locked — do not use until evaluation")

 Train/test split complete
  Train set:  25,508 patients (80%)
  Test set:   6,377 patients  (20%)

  Train mortality rate: 10.6%
  Test mortality rate:  10.6%

   Test set is now locked — do not use until evaluation


In [10]:
# Normalise continuous features using Quantile Transformer
# Quantile normalisation maps values to a uniform or normal
# distribution — robust to skewed lab distributions and outliers
# TabDDPM and CTGAN both perform better on normalised data

continuous_cols = ['AGE', 'LOS'] + lab_cols

scaler = QuantileTransformer(
    output_distribution='normal',
    n_quantiles=1000,
    random_state=42
)

# Fit ONLY on training data — never on test data
train_df[continuous_cols] = scaler.fit_transform(
    train_df[continuous_cols]
)

# Transform test data using the same fitted scaler
test_df[continuous_cols] = scaler.transform(
    test_df[continuous_cols]
)

print(f" Normalisation complete")
print(f"  Method: Quantile Transformer (normal output)")
print(f"  Columns normalised: {len(continuous_cols)}")
print(f"\nSample — AGE after normalisation:")
print(f"  Mean:   {train_df['AGE'].mean():.3f}  (should be ~0)")
print(f"  Std:    {train_df['AGE'].std():.3f}   (should be ~1)")
print(f"  Min:    {train_df['AGE'].min():.3f}")
print(f"  Max:    {train_df['AGE'].max():.3f}")

 Normalisation complete
  Method: Quantile Transformer (normal output)
  Columns normalised: 14

Sample — AGE after normalisation:
  Mean:   -0.004  (should be ~0)
  Std:    0.997   (should be ~1)
  Min:    -5.199
  Max:    5.199


In [11]:
# Save the scaler so we can reverse-transform
# synthetic data back to original scale after generation
import pickle

MODELS_PATH = r"\models"
os.makedirs(MODELS_PATH, exist_ok=True)

scaler_path = os.path.join(MODELS_PATH, 'quantile_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f" Scaler saved → {scaler_path}")
print(f"  This scaler must be used to reverse-transform")
print(f"  synthetic data back to clinical units after generation")

 Scaler saved → \models\quantile_scaler.pkl
  This scaler must be used to reverse-transform
  synthetic data back to clinical units after generation


In [12]:
# Save final datasets
train_df.to_csv(
    os.path.join(PROCESSED_PATH, 'train_real.csv'), index=False
)
test_df.to_csv(
    os.path.join(PROCESSED_PATH, 'test_real.csv'), index=False
)

print(f" train_real.csv saved → {PROCESSED_PATH}")
print(f" test_real.csv  saved → {PROCESSED_PATH}")
print(f"\nFinal summary:")
print(f"  Train shape: {train_df.shape}")
print(f"  Test shape:  {test_df.shape}")
print(f"  Features:    {train_df.shape[1] - 1}")
print(f"  Target:      {target_col}")

 train_real.csv saved → data\processed
 test_real.csv  saved → data\processed

Final summary:
  Train shape: (25508, 34)
  Test shape:  (6377, 34)
  Features:    33
  Target:      HOSPITAL_EXPIRE_FLAG


In [13]:
# Final sanity check on both sets
print("=" * 50)
print("       PREPROCESSING COMPLETE — FINAL CHECK")
print("=" * 50)

for name, dataset in [("TRAIN", train_df), ("TEST", test_df)]:
    print(f"\n{name} SET:")
    print(f"  Rows:          {len(dataset):,}")
    print(f"  Columns:       {dataset.shape[1]}")
    print(f"  Null values:   {dataset.isnull().sum().sum()}")
    print(f"  Mortality rate:{dataset[target_col].mean()*100:.1f}%")
    print(f"  AGE mean:      {dataset['AGE'].mean():.3f}")
    print(f"  GENDER (% M):  {dataset['GENDER'].mean()*100:.1f}%")

print("\n" + "=" * 50)
print("   Ready for CTGAN and TabDDPM training")
print("=" * 50)

       PREPROCESSING COMPLETE — FINAL CHECK

TRAIN SET:
  Rows:          25,508
  Columns:       34
  Null values:   0
  Mortality rate:10.6%
  AGE mean:      -0.004
  GENDER (% M):  58.0%

TEST SET:
  Rows:          6,377
  Columns:       34
  Null values:   0
  Mortality rate:10.6%
  AGE mean:      -0.031
  GENDER (% M):  57.4%

   Ready for CTGAN and TabDDPM training


## Notebook 03 — Preprocessing Complete

### Steps Performed
| Step | Action | Why |
|------|--------|-----|
| 1 | Outlier removal | Clinical bounds filter impossible values |
| 2 | Missingness indicators | Preserve clinically informative absence |
| 3 | Median imputation | Fill remaining gaps robustly |
| 4 | Categorical encoding | Models need numbers not text |
| 5 | Train/test split (80/20) | Lock test set before any modelling |
| 6 | Quantile normalisation | Stabilise skewed lab distributions |

### Output Files
- `train_real.csv` — 80% of cohort, normalised, for model training
- `test_real.csv`  — 20% of cohort, locked until evaluation
- `quantile_scaler.pkl` — fitted scaler for reverse transformation


In [14]:
import pandas as pd
import os

PROCESSED_PATH = r"data\processed"

train = pd.read_csv(os.path.join(PROCESSED_PATH, 'train_real.csv'))
test  = pd.read_csv(os.path.join(PROCESSED_PATH, 'test_real.csv'))

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Train nulls: {train.isnull().sum().sum()}")
print(f"Test nulls:  {test.isnull().sum().sum()}")
print(f"Columns: {list(train.columns)}")

Train shape: (25508, 34)
Test shape:  (6377, 34)
Train nulls: 0
Test nulls:  0
Columns: ['AGE', 'GENDER', 'LOS', 'lab_base_excess', 'lab_bicarbonate', 'lab_bun', 'lab_creatinine', 'lab_glucose', 'lab_haemoglobin', 'lab_lactate', 'lab_ph', 'lab_platelets', 'lab_potassium', 'lab_sodium', 'lab_wbc', 'lab_base_excess_missing', 'lab_bicarbonate_missing', 'lab_bun_missing', 'lab_creatinine_missing', 'lab_glucose_missing', 'lab_haemoglobin_missing', 'lab_lactate_missing', 'lab_ph_missing', 'lab_platelets_missing', 'lab_potassium_missing', 'lab_sodium_missing', 'lab_wbc_missing', 'admtype_EMERGENCY', 'admtype_URGENT', 'insurance_Medicaid', 'insurance_Medicare', 'insurance_Private', 'insurance_Self Pay', 'HOSPITAL_EXPIRE_FLAG']
